# Lưu cache prompt full từ đúng Output175
**Accelerator None, Internet ON; không cần Add Input.** Bật Secret `codex` của tài khoản sở hữu Output175. Tải đúng175 và kiểm14fileSHA cùng vectors410×512/dtype/norm/cohort trước khi tạo **Dataset Private mới**. Không chạy lại cache/CLIP hoặc load caption checkpoint. Không dùng latest; không xóa/ghi đè output. Sau khi Dataset xử lý xong hãy chia sẻ cho Ngân qua quyền Dataset, không chia sẻ token.


In [ ]:
from pathlib import Path
OWNER='ducanh2403'
SLUG='csa-prompt-full-karpathy-v175'
TITLE='CSA full Karpathy prompt cache v175'
SECRET_NAME='codex'
DESTINATION=Path('/kaggle/working/csa_prompt_full_karpathy_v175_dataset')


In [ ]:
import json,os,re,subprocess,sys,tempfile,shutil
from kaggle_secrets import UserSecretsClient
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('Title/slug must have6–50characters')
if DESTINATION.exists():raise RuntimeError('Preserve existing output; choose fresh DESTINATION')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<4*1024**3:raise RuntimeError('Need4GiB free for download/copy; no files removed')
try:token=UserSecretsClient().get_secret(SECRET_NAME).strip()
except Exception:raise RuntimeError('Enable the named Secret for the owner of private output175') from None
if not token:raise RuntimeError('Empty API token')
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(k,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    process=subprocess.Popen(list(map(str,args)),env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace')
    for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
    if process.wait():raise RuntimeError('API command failed; preserve staged files and read redacted log above')
CLI=[sys.executable,'-c','from kaggle.cli import main; main()']
run_cli([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],authenticated=False)


In [ ]:
VERSION_PREFLIGHT="\nimport tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nfor version in ('175',):\n    with tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n        endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n        endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n        api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/'+version,path=directory)\n        request=endpoint.call_args.args[0]\n        assert request.version_label=='v'+version and request.user_name=='ducanh2403' and request.kernel_slug=='2-vi-cai-kia-qua-nhieu','Exact version ignored'\nprint('Exact175 API routing preflight PASS; no network in this check.',flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib
INVENTORY={'cache_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/175', 'cache_files': {'cohort.json': 'd97eaa8cecd07a4de825e3be99926f751f610e103ff9582c2b482e53d2c9980d', 'prompt_texts.json': '35f28b5f9f6cbbc2eaa6a0fc3ea816f8722013a5804814da6555edcfe97f04c2', 'prompt_links.jsonl': 'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0', 'summary.json': '28d650c5403fcf851e9a8f5883824ceba2d46144a9d2222be36cb056e07b5122', 'review_examples.json': '17b4ed35145338ac3fe65b28b293e4a11d8ac39fba1199ff3a46b29219329df7', 'prompt_embeddings.npy': 'f59b4b08c578e98195f68c2af7899a181c83678e406ddbf3a661ba36cc87e98c', 'mapping_receipt.json': '134c64c5c018e37fc138c98a6ce59e8a8b306433d20c53c9559b781683eaf09d', 'reuse_report.json': 'de76c927e7839bfb553ca09ea3373474ba2afeae6b12b7f1e9959d8ed41358ef', 'cache_manifest.json': '05f40a246b19cb37717a4aadb7bcb20f2e0dd0ebabe60e983d83b9f328e9f56a', 'cache_receipt.json': '72c532cd73b832c5cdf8f0ff747a882413e006fa84670f61a6a54abd272613cb', 'run_status.json': 'fc1f7837f14785888c4ff30c193ca71b72ce34931f93c918cc23a9945089d8e9', 'source_receipt.json': '7df1b07aa9e8ef821e4608f406c355142d3435dbd7aa5bbf805daf4e6afd5365', 'notebook.log': '5c656006b117092d6cc19ef472d18e7fb264a6f1a00876be16f694af87facd74', 'cache_plan.json': '4f1fd4c729fe72db06de781ad2a99af8755eef225cae23ecd989778438aeb057'}, 'source_review_zip_sha256': 'c843884773bffe56a2ae66ef4ffc0f62002881602ade439707d56a276850775c'}
SOURCE_CODE='"""Byte-preserving CPU transport for exact169 cache and exact161 weights; no models."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport numpy as np\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef pattern(names):\n    return r\'(^|/)(\'+\'|\'.join(re.escape(n) for n in names)+r\')$\'\n\n\ndef locate(root,name,expected):\n    matches=[p for p in Path(root).rglob(name) if p.is_file() and sha(p)==expected]\n    if len(matches)!=1:\n        raise ValueError(f\'Need one exact {name}; found {len(matches)} matching SHA, no version fallback\')\n    return matches[0]\n\n\ndef package_full_cache(download,destination,inventory):\n    """Preserve exact175 fresh cache, without caption weights or neural imports."""\n    destination=Path(destination);files=inventory[\'cache_files\']\n    if inventory[\'cache_version\']!=\'ducanh2403/2-vi-cai-kia-qua-nhieu/175\':raise ValueError(\'Expected exact175\')\n    if destination.exists():raise FileExistsError(\'Preserve existing destination\')\n    if any(Path(n).name!=n or \'/\' in n or \'\\\\\' in n for n in files):raise ValueError(\'Unsafe payload name\')\n    root=locate(download,\'cache_receipt.json\',files[\'cache_receipt.json\']).parent\n    for n,h in files.items():\n        if not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed175 payload:\'+n)\n    read=lambda n:json.loads((root/n).read_text(encoding=\'utf-8\'))\n    receipt=read(\'cache_receipt.json\');manifest=read(\'cache_manifest.json\');state=read(\'run_status.json\')\n    expected={\'cohort.json\',\'prompt_texts.json\',\'prompt_links.jsonl\',\'summary.json\',\'review_examples.json\',\'prompt_embeddings.npy\',\'mapping_receipt.json\',\'reuse_report.json\',\'cache_manifest.json\'}\n    if receipt[\'status\']!=\'FULL_PROMPT_TEXT_CACHE_COMPLETE\' or set(receipt[\'payload_sha256\'])!=expected or any(files.get(n)!=h for n,h in receipt[\'payload_sha256\'].items()):raise ValueError(\'Full cache receipt contract differs\')\n    if state!=dict(complete=True,training=False,test_used=False,caption_weights_loaded=False):raise ValueError(\'Wrong cache run status\')\n    if manifest[\'status\']!=\'FULL_PROMPT_TEXT_CACHE_COMPLETE\' or manifest[\'cache_mode\']!=\'fresh_all\' or manifest[\'shape\']!=[410,512] or manifest[\'training\'] or manifest[\'test_used\']:raise ValueError(\'Wrong full cache manifest\')\n    texts=read(\'prompt_texts.json\');cohort=read(\'cohort.json\');vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if len(texts)!=410 or texts!=sorted(set(texts)) or any(not isinstance(t,str) or not t.strip() for t in texts):raise ValueError(\'Wrong full text vocabulary\')\n    if vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError(\'Embedding shape/dtype/norm differs\')\n    if len(cohort[\'train\'])!=len(set(cohort[\'train\'])) or len(cohort[\'train\'])!=113287 or len(cohort[\'val\'])!=len(set(cohort[\'val\'])) or len(cohort[\'val\'])!=5000 or cohort[\'test\'] or set(cohort[\'train\'])&set(cohort[\'val\']):raise ValueError(\'Full cohort coverage differs\')\n    destination.mkdir(parents=True)\n    for n,h in files.items():\n        shutil.copy2(root/n,destination/n)\n        if sha(destination/n)!=h:raise ValueError(\'Copied bytes changed:\'+n)\n    report=dict(status=\'FULL_PROMPT_V175_PRESERVED\',cache_source=inventory[\'cache_version\'],files_sha256=files,\n        source_review_zip_sha256=inventory[\'source_review_zip_sha256\'],original_bytes=True,embedding_shape=[410,512],\n        models_executed=False,training=False,test_used=False)\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n\n\ndef package(cache_download,weight_download,destination,inventory):\n    destination=Path(destination)\n    if destination.exists():raise FileExistsError(\'Preserve existing destination; choose a fresh one\')\n    files=inventory[\'cache_files\']\n    if any(Path(n).name!=n or \'/\' in n or \'\\\\\' in n for n in files):raise ValueError(\'Unsafe payload name\')\n    receipt_path=locate(cache_download,\'cache_receipt.json\',files[\'cache_receipt.json\'])\n    root=receipt_path.parent\n    receipt=json.loads(receipt_path.read_text(encoding=\'utf-8\'))\n    if receipt[\'status\']!=\'PROMPT_TEXT_CACHE_COMPLETE\':raise ValueError(\'Incomplete cache\')\n    if any(n not in files or files[n]!=h for n,h in receipt[\'payload_sha256\'].items()):raise ValueError(\'Cache receipt identities differ\')\n    if set(receipt[\'payload_sha256\'])!={\'prompt_embeddings.npy\',\'prompt_texts.json\',\'prompt_links.jsonl\',\'cohort.json\',\'summary.json\',\'cache_manifest.json\'}:\n        raise ValueError(\'Cache receipt payload coverage differs\')\n    for n,h in files.items():\n        if not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed169 payload:\'+n)\n    state=json.loads((root/\'run_status.json\').read_text(encoding=\'utf-8\'))\n    if not state[\'complete\'] or state[\'training\'] or state[\'test_used\']:raise ValueError(\'Wrong cache run status\')\n    vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if vectors.shape!=(318,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):\n        raise ValueError(\'Embedding shape/dtype/norm differs\')\n    weight=locate(weight_download,\'best.bin\',inventory[\'checkpoint_sha256\'])\n    # Binary identity is checked without deserializing weights or importing Torch.\n    destination.mkdir(parents=True)\n    for n in files:shutil.copy2(root/n,destination/n)\n    shutil.copy2(weight,destination/\'csa_ce161_best.bin\')\n    expected={**files,\'csa_ce161_best.bin\':inventory[\'checkpoint_sha256\']}\n    for n,h in expected.items():\n        if sha(destination/n)!=h:raise ValueError(\'Copied bytes changed:\'+n)\n    report=dict(status=\'PROMPT_PILOT_INPUTS_PRESERVED\',cache_source=inventory[\'cache_version\'],\n        checkpoint_source=inventory[\'checkpoint_version\'],files_sha256=expected,\n        original_bytes=True,models_executed=False,training=False,test_used=False)\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='2d20b00d009a2d95525ec311602db84716d499ec14eb26bd6aa653fb3ee80523'
exec(compile(SOURCE_CODE,'package_prompt_inputs.py','exec'))


In [ ]:
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa-prompt-full175-',dir='/tmp'))
run_cli(CLI+['kernels','output',INVENTORY['cache_version'],'-p',str(DOWNLOAD),'--file-pattern',pattern(INVENTORY['cache_files'])])
report=package_full_cache(DOWNLOAD,DESTINATION,INVENTORY)
print('Verified and copied original175 bytes:',len(report['files_sha256']),'files;410x512 vectors. No model loaded.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; use fresh destination')
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private frozen CLIP prompt cache from exact output175: full Karpathy TRAIN113287 and VALID5000, 410 unique text vectors linked to ROI134. No TEST, images or trained caption weights. Upstream data/model terms retained.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be active. Check page before retrying.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
print('After processing, grant Ngan access in Dataset sharing; do not give anyone your API token.',flush=True)


Gửi path mount khi xử lý xong. Dự kiến `/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175`. Đây chỉ là cache prompt; ROI134 và semantic banks130 vẫn gắn riêng khi huấn luyện từ đầu. Không cần checkpoint161/CSA cũ.
